# Creating corrupted images


In [ ]:
# ─────────────────────────────────────────────────────────────
# Grid-by-Grid Image Corruption Script — Group 10 IDS 705
# Sebine Scaria
# 224x224 PneumoniaMNIST
# ─────────────────────────────────────────────────────────────

# ── Step 1: Mount Drive and install ──────────────────────────
import shutil

# Clear the mount point first
shutil.rmtree('/content/drive', ignore_errors=True)

# Now remount
from google.colab import drive
drive.mount('/content/drive')

!pip install medmnist --quiet

import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import numpy as np
import os
import io
from PIL import Image
from medmnist import INFO, PneumoniaMNIST
from torch.utils.data import DataLoader

print('Libraries loaded!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries loaded!


In [ ]:
# ── Step 2: Save paths ────────────────────────────────────────
# ── Run this every time Colab resets ─────────────────────────
BASE_DIR      = '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)'
CLEAN_DIR     = os.path.join(BASE_DIR, 'Clean_images')
CORRUPTED_DIR = '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images'

os.makedirs(CORRUPTED_DIR, exist_ok=True)

print(f'Clean dir:     {CLEAN_DIR}')
print(f'Corrupted dir: {CORRUPTED_DIR}')

Clean dir:     /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Clean_images
Corrupted dir: /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images


In [ ]:
# ── Step 3: Load datasets ─────────────────────────────────────
SIZE = 224

data_transform = transforms.Compose([
    transforms.Resize((SIZE, SIZE), interpolation=Image.NEAREST),
    transforms.Lambda(lambda img: img.convert('L')),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True, size=SIZE)
val_dataset   = PneumoniaMNIST(split='val',   transform=data_transform, download=True, size=SIZE)
test_dataset  = PneumoniaMNIST(split='test',  transform=data_transform, download=True, size=SIZE)

def load_all(dataset):
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    images, labels = next(iter(loader))
    return images, labels

train_images, train_labels = load_all(train_dataset)
val_images,   val_labels   = load_all(val_dataset)
test_images,  test_labels  = load_all(test_dataset)

print(f'Train: {train_images.shape}')
print(f'Val:   {val_images.shape}')
print(f'Test:  {test_images.shape}')

Train: torch.Size([4708, 1, 224, 224])
Val:   torch.Size([524, 1, 224, 224])
Test:  torch.Size([624, 1, 224, 224])


In [ ]:
# ── Step 4: Attack functions at L3 ───────────────────────────

def attack_brightness_contrast(region):
    """L3: Brightness dark delta=-0.3, Contrast low alpha=0.4"""
    region = torch.clamp(region - 0.3, -1, 1)
    region = torch.clamp(region * 0.4, -1, 1)
    return region

def attack_jpeg(region):
    """L3: JPEG quality=30"""
    results = []
    for img in region:
        img_01     = (img * 0.5 + 0.5).clamp(0, 1)
        img_pil    = TF.to_pil_image(img_01.squeeze(0))
        buf        = io.BytesIO()
        img_pil.save(buf, format='JPEG', quality=30)
        buf.seek(0)
        img_back   = Image.open(buf).copy()
        img_tensor = TF.to_tensor(img_back).unsqueeze(0)
        results.append((img_tensor - 0.5) / 0.5)
    return torch.cat(results, dim=0)

def attack_downsampling(region):
    """L3: scale=0.3"""
    _, _, H, W = region.shape
    small_h    = max(1, int(H * 0.3))
    small_w    = max(1, int(W * 0.3))
    down = F.interpolate(region, size=(small_h, small_w), mode='nearest')
    return F.interpolate(down, size=(H, W), mode='nearest')

def attack_gaussian_blur(region):
    """L3: kernel=11, sigma=2.0"""
    return TF.gaussian_blur(region, kernel_size=11, sigma=2.0)

ATTACK_NAMES = [
    'brightness_contrast',
    'jpeg_compression',
    'downsampling',
    'gaussian_blur'
]
print(f'4 attacks defined at L3: {ATTACK_NAMES}')

4 attacks defined at L3: ['brightness_contrast', 'jpeg_compression', 'downsampling', 'gaussian_blur']


In [ ]:
# ── Step 5: Grid bounds helper ────────────────────────────────
def get_grid_bounds(H, W, row, col):
    h_start = row * H // 3
    h_end   = (row + 1) * H // 3
    w_start = col * W // 3
    w_end   = (col + 1) * W // 3
    return h_start, h_end, w_start, w_end

In [ ]:
# ── Step 6: Generate and save (low memory) ────────────────────
def generate_and_save_lowmem(dataset, split_name, save_path):
    print(f'\nCorrupting {split_name} set...')
    N = len(dataset)

    attack_fns = [
        attack_brightness_contrast,
        attack_jpeg,
        attack_downsampling,
        attack_gaussian_blur
    ]

    all_corrupted  = []
    all_labels     = []
    all_attack_ids = []
    all_grid_pos   = []

    for i in range(N):
        if i % 100 == 0:
            print(f'  Image {i}/{N}...')

        img, label = dataset[i]
        img = img.unsqueeze(0)
        H, W = img.shape[2], img.shape[3]

        for row in range(3):
            for col in range(3):
                h_start, h_end, w_start, w_end = get_grid_bounds(H, W, row, col)
                for attack_id, attack_fn in enumerate(attack_fns):
                    copy = img.clone()
                    copy[:, :, h_start:h_end, w_start:w_end] = attack_fn(
                        img[:, :, h_start:h_end, w_start:w_end]
                    )
                    all_corrupted.append(copy)
                    all_labels.append(torch.tensor(label).unsqueeze(0))
                    all_attack_ids.append(attack_id)
                    all_grid_pos.append((row, col))

    print('  Stacking tensors...')
    corrupted_images = torch.cat(all_corrupted, dim=0)
    corrupted_labels = torch.cat(all_labels,    dim=0)
    attack_ids       = torch.tensor(all_attack_ids)

    torch.save({
        'images'        : corrupted_images,
        'labels'        : corrupted_labels,
        'attack_ids'    : attack_ids,
        'attack_names'  : ATTACK_NAMES,
        'grid_positions': all_grid_pos,
        'split'         : split_name,
        'size'          : SIZE,
        'severity'      : 'L3',
        'note'          : f'Grid-corrupted 224x224 {split_name} — 4 attacks x 9 grid positions L3'
    }, save_path)

    print(f'  ✅ {corrupted_images.shape[0]} images saved → {save_path}')

# Session 1 — train only
generate_and_save_lowmem(train_dataset,
    'train', os.path.join(CORRUPTED_DIR, 'train_corrupted.pt'))


Corrupting train set...
  Image 0/4708...
  Image 100/4708...
  Image 200/4708...
  Image 300/4708...
  Image 400/4708...
  Image 500/4708...
  Image 600/4708...
  Image 700/4708...
  Image 800/4708...
  Image 900/4708...
  Image 1000/4708...
  Image 1100/4708...
  Image 1200/4708...
  Image 1300/4708...
  Image 1400/4708...
  Image 1500/4708...
  Image 1600/4708...
  Image 1700/4708...
  Image 1800/4708...
  Image 1900/4708...
  Image 2000/4708...
  Image 2100/4708...
  Image 2200/4708...
  Image 2300/4708...
  Image 2400/4708...
  Image 2500/4708...
  Image 2600/4708...
  Image 2700/4708...
  Image 2800/4708...
  Image 2900/4708...
  Image 3000/4708...
  Image 3100/4708...
  Image 3200/4708...
  Image 3300/4708...
  Image 3400/4708...
  Image 3500/4708...
  Image 3600/4708...
  Image 3700/4708...
  Image 3800/4708...
  Image 3900/4708...
  Image 4000/4708...
  Image 4100/4708...
  Image 4200/4708...
  Image 4300/4708...
  Image 4400/4708...
  Image 4500/4708...
  Image 4600/4708...


In [ ]:
generate_and_save_lowmem(val_dataset,
    'val', os.path.join(CORRUPTED_DIR, 'val_corrupted.pt'))


Corrupting val set...
  Image 0/524...
  Image 100/524...
  Image 200/524...
  Image 300/524...
  Image 400/524...
  Image 500/524...
  Stacking tensors...
  ✅ 18864 images saved → /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/val_corrupted.pt


In [ ]:
import os
fpath = os.path.join(CORRUPTED_DIR, 'val_corrupted.pt')
if os.path.exists(fpath):
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f'✅ val_corrupted.pt — {size_mb:.1f} MB — still exists')
else:
    print(f'❌ val_corrupted.pt — NOT FOUND')

✅ val_corrupted.pt — 3611.2 MB — still exists
